#Task 5: Implementing SVD for Recommendations

In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split

# 1. Load Data
r_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings = pd.read_csv('u.data', sep='\t', names=r_cols, encoding='latin-1')

m_cols = ['movie_id', 'movie_title']
movies = pd.read_csv('u.item', sep='|', names=m_cols, usecols=[0, 1], encoding='latin-1')

# 2. Train/Test Split
train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)

# 3. Create User-Item Matrix
user_item_matrix = train_data.pivot(index='user_id', columns='movie_id', values='rating').fillna(0)

print(f"Original User-Item Matrix Shape: {user_item_matrix.shape}")

Original User-Item Matrix Shape: (943, 1653)


In [2]:
svd = TruncatedSVD(n_components=20, random_state=42)
latent_matrix = svd.fit_transform(user_item_matrix)
reconstructed_matrix = np.dot(latent_matrix, svd.components_)

preds_df = pd.DataFrame(reconstructed_matrix,
                        columns=user_item_matrix.columns,
                        index=user_item_matrix.index)

print("Matrix Factorization complete. Predictions generated for all missing ratings.")
preds_df.head()

Matrix Factorization complete. Predictions generated for all missing ratings.


movie_id,1,2,3,4,5,6,7,8,9,10,...,1668,1670,1671,1672,1673,1676,1678,1679,1680,1681
user_id,,,,,,,,,,,,,,,,,,,,,
1,2.358825,1.498288,1.430347,1.553020,0.134955,0.319592,4.400575,1.784258,3.007316,2.766778,...,-0.002292,-0.002292,-0.003988,0.032542,-0.020922,-0.000996,0.000471,0.001412,0.000941,0.006383
2,1.635280,-0.207181,0.048970,0.358840,-0.185678,0.293039,1.545101,0.124580,1.974953,0.464599,...,0.013881,0.013881,0.003511,0.003852,0.008304,-0.007543,0.005332,0.015996,0.010664,-0.002509
3,-0.216737,-0.097211,0.152692,0.054688,0.011917,0.016639,-0.120750,0.275699,-0.155474,0.045694,...,0.031209,0.031209,0.005734,0.007271,0.002599,0.013594,0.012053,0.036159,0.024106,-0.001453
4,0.195015,-0.278614,0.033653,0.226117,0.028011,-0.079217,0.071664,-0.003163,-0.259782,-0.042696,...,0.022580,0.022580,0.005222,0.003272,-0.002065,0.003422,0.006894,0.020681,0.013787,0.001918
5,2.529358,0.937370,0.248939,0.806282,0.348005,-0.111758,2.425996,0.988594,-0.498795,0.171913,...,-0.006721,-0.006721,-0.001249,0.005454,-0.011840,-0.015602,-0.003978,-0.011933,-0.007955,0.012412


In [3]:
def get_svd_recommendations(user_id, n=10):
    if user_id not in preds_df.index:
        return []

    # 1. Get the user's predicted ratings
    user_predictions = preds_df.loc[user_id]

    # 2. Get the movies the user has already seen (to filter them out)
    user_seen = set(train_data[train_data['user_id'] == user_id]['movie_id'])

    # 3. Sort predictions descending
    sorted_preds = user_predictions.sort_values(ascending=False)

    # 4. Filter out seen movies and return Top N
    recommendations = [m_id for m_id in sorted_preds.index if m_id not in user_seen]

    return recommendations[:n]

test_user = 1
top_svd_picks = get_svd_recommendations(test_user, n=5)

print(f"SVD Recommendations for User {test_user} (Movie IDs):")
print(top_svd_picks)

# Map back to titles
print("\nMovie Titles:")
print(movies[movies['movie_id'].isin(top_svd_picks)]['movie_title'].tolist())

SVD Recommendations for User 1 (Movie IDs):
[475, 276, 433, 186, 652]

Movie Titles:
['Blues Brothers, The (1980)', 'Leaving Las Vegas (1995)', 'Heathers (1989)', 'Trainspotting (1996)', 'Rosencrantz and Guildenstern Are Dead (1990)']


In [4]:
def evaluate_svd(n_users=100, k=10):
    all_precision = []
    all_recall = []

    common_users = set(train_data['user_id']).intersection(set(test_data['user_id']))

    for user_id in list(common_users)[:n_users]:
        # Ground Truth from Test Set (Rating >= 4)
        actual_relevant = set(test_data[(test_data['user_id'] == user_id) & (test_data['rating'] >= 4)]['movie_id'])

        if len(actual_relevant) == 0:
            continue

        # Predictions from SVD
        recommended = set(get_svd_recommendations(user_id, n=k))

        if len(recommended) == 0:
            continue

        # Metrics Calculation
        hits = len(recommended.intersection(actual_relevant))
        precision = hits / k
        recall = hits / len(actual_relevant)

        all_precision.append(precision)
        all_recall.append(recall)

    avg_p = np.mean(all_precision)
    avg_r = np.mean(all_recall)
    f1 = (2 * avg_p * avg_r) / (avg_p + avg_r) if (avg_p + avg_r) > 0 else 0

    return avg_p, avg_r, f1

# Execute evaluation
prec_svd, rec_svd, f1_svd = evaluate_svd(n_users=200, k=10)

print("--- Task 5: SVD Matrix Factorization Metrics ---")
print(f"Average Precision @ 10: {prec_svd:.4f}")
print(f"Average Recall @ 10:    {rec_svd:.4f}")
print(f"F1-Score:               {f1_svd:.4f}")

--- Task 5: SVD Matrix Factorization Metrics ---
Average Precision @ 10: 0.2503
Average Recall @ 10:    0.2777
F1-Score:               0.2633


#Task 6: Implementing Matrix Factorization with the Surprise Library

In [1]:
!pip install "numpy<2"
!pip install scikit-surprise

In [2]:
import pandas as pd
from surprise import Reader, Dataset, SVD, accuracy
from surprise.model_selection import train_test_split, cross_validate

r_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings = pd.read_csv('u.data', sep='\t', names=r_cols, encoding='latin-1')

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings[['user_id', 'movie_id', 'rating']], reader)

print("Data successfully loaded into the Surprise Dataset format.")

Data successfully loaded into the Surprise Dataset format.


In [3]:
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

algo = SVD(n_factors=50, random_state=42)

algo.fit(trainset)

print("SVD Model trained successfully using the Surprise library.")

SVD Model trained successfully using the Surprise library.


In [5]:
# 1. Generate predictions on the hidden test set
predictions = algo.test(testset)

# 2. Calculate accuracy metrics
print("--- Task 6: Model Evaluation ---")
rmse = accuracy.rmse(predictions, verbose=True)
mae = accuracy.mae(predictions, verbose=True)

--- Task 6: Model Evaluation ---
RMSE: 0.9348
MAE:  0.7377


In [6]:
uid = 1
iid = 50

prediction = algo.predict(uid, iid)

print(f"Prediction for User {uid} on Movie {iid}:")
print(f"Estimated Rating: {prediction.est:.2f} stars")

Prediction for User 1 on Movie 50:
Estimated Rating: 4.69 stars
